# GPU benchmarks on Google Colab

This notebook builds, runs, profiles and plots the GPU (CUDA) versions of the parallel binary search tree search.

**Runtime → Change runtime type → T4 GPU** before running it. The published results were obtained on an NVIDIA Tesla T4 (compute capability 7.5, hence `-arch=sm_75` in the `Makefile`).

Steps:
1. Get the project (clone from GitHub or unzip from Google Drive) and check the GPU
2. Build the executables
3. Run the benchmarks
4. Profile with Nsight Systems / Nsight Compute
5. Plot the results
6. (Optional) Save the results to Google Drive

## 1 - Setup

Get the project into the Colab machine in **one** of the two ways below, then continue from the GPU check.

#### Option A - Clone from GitHub

In [ ]:
# Set this to the URL of the repository
REPO_URL = "https://github.com/gabrielecaioli/cpu-gpu-bottleneck-analysis.git"
REPO_DIR = "/content/repo"

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

#### Option B - From a zip file on Google Drive

Useful to test the code before publishing it. On your computer, from the folder that contains the project, create the zip
(executables and build files are not needed):

```
zip -r CAproject.zip CAproject -x "*/bin/*" "*/build/*" "*/.git/*"
```

upload `CAproject.zip` to Google Drive and set `ZIP_PATH` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path of the zip file on Google Drive
ZIP_PATH = "/content/drive/MyDrive/CAproject.zip"
UNZIP_DIR = "/content/project"

!unzip -o -q "{ZIP_PATH}" -d {UNZIP_DIR}

# The project folder is the one that contains the Makefile
import glob, os
makefiles = [m for m in glob.glob(f"{UNZIP_DIR}/**/Makefile", recursive=True) if "__MACOSX" not in m]
REPO_DIR = os.path.dirname(min(makefiles, key=len))
%cd {REPO_DIR}
!ls

#### GPU check

In [ ]:
!nvidia-smi
!nvcc --version

#### GPU information
Hardware limits used to choose the number of threads: e.g. a Tesla T4 has 40 SMs with up to 1024 resident threads each,
so at most 40 × 1024 = 40960 threads run concurrently.

In [ ]:
from numba import cuda

device = cuda.get_current_device()

def print_attr(attr_name, label=None, transform=lambda x: x):
    """Prints a device attribute, if available."""
    if hasattr(device, attr_name):
        value = getattr(device, attr_name)
        try:
            value = transform(value)
        except Exception:
            value = str(value)
        print(f"{label or attr_name}: {value}")

print_attr("name", "Device name")
print_attr("compute_capability", "Compute Capability")
print_attr("WARP_SIZE", "Threads per warp")
print_attr("MAX_THREADS_PER_BLOCK", "Max threads per block")
print_attr("MAX_GRID_DIM_X", "Max blocks per grid - X")
print_attr("MAX_BLOCK_DIM_X", "Max threads per dimension - X")
print_attr("MAX_SHARED_MEMORY_PER_BLOCK", "Shared memory per block", lambda x: f"{x // 1024} KB")
print_attr("MAX_CONSTANT_MEMORY", "Total constant memory", lambda x: f"{x // 1024} KB")
print_attr("MAX_REGISTERS_PER_BLOCK", "Registers per block")
print_attr("CLOCK_RATE", "Clock rate", lambda x: f"{x / 1e6:.2f} GHz")
print_attr("MEMORY_CLOCK_RATE", "Memory clock rate", lambda x: f"{x / 1e6:.2f} GHz")
print_attr("GLOBAL_MEMORY_BUS_WIDTH", "Global memory bus width", lambda x: f"{x} bit")
print_attr("TOTAL_GLOBAL_MEM", "Total global memory", lambda x: f"{x / (1024 ** 3):.2f} GB")
print_attr("MULTIPROCESSOR_COUNT", "Multiprocessor count")

## 2 - Build

### 2.1 - Quick test

Builds `bin/gpu_v1`, `bin/gpu_v2_1`, `bin/gpu_v2_2` and `bin/gpu_v3` with a small dataset (1,000,000 values to search)
and runs each version once, printing its timing breakdown. Takes about a minute: run this first to check that everything works.

Arguments of the executables:
`<THREADS_PER_BLOCK> <NUM_BLOCKS> <NUM_NODES> <NUM_TO_SEARCH> <TO_EXTRACT> <VERBOSE> <VERBOSE_TIMES> <REPETITIONS_OF_WORKLOAD>`

In [ ]:
!make gpu QUERIES=1000000

for version, threads_per_block in [("v1", 32), ("v2_1", 32), ("v2_2", 32), ("v3", 1024)]:
    num_blocks = 32768 // threads_per_block   # 2^15 threads in total
    !./bin/gpu_{version} {threads_per_block} {num_blocks} 8190 1000000 0 0 1 1
    print()

### 2.2 - Full build (optional)

Same executables with the full dataset used for the published results (500,000 values to insert, 10,000,000 values to search).
Generating and compiling it takes a few minutes. Not needed before section 3: the benchmark script builds its own executables.

In [ ]:
!make gpu

## 3 - Benchmarks

`scripts/benchmark_gpu.py` builds one executable per dataset seed (`bin/gpu_<version>_<seed>`), runs every configuration once per seed
and saves the parsed timings in `results/gpu/res_<version>_w<workload>_TpB<threads per block>.json`.

With the defaults (30 seeds, 4 workloads, 14 thread counts) a full run takes a long time: reduce `--seeds`, `--queries` or `--threads` to try it out.

In [ ]:
# Versions 1, 2.1 and 2.2 with 32 threads per block
!python3 scripts/benchmark_gpu.py --versions v1 v2_1 v2_2 --threads-per-block 32

In [ ]:
# Version 3 with 1024 threads per block
!python3 scripts/benchmark_gpu.py --versions v3 --threads-per-block 1024

In [ ]:
# Optional: effect of the number of threads per block on Version 3 (10M values to search)
for tpb in [32, 64, 128, 256, 512]:
    !python3 scripts/benchmark_gpu.py --versions v3 --queries 10000000 --threads-per-block {tpb} --no-build

## 4 - Profiling

Profiles of a single run, to be opened with Nsight Systems (`.nsys-rep`) and Nsight Compute (`.ncu-rep`).

In [ ]:
VERSION = "v3"
ARGS = "1024 32 8190 10000000 0 0 0 1"   # see the arguments of the quick test above

!mkdir -p profiling
!nsys profile -o profiling/prof_{VERSION} --force-overwrite true ./bin/gpu_{VERSION} {ARGS} > /dev/null 2>&1
!ncu -f -o profiling/prof_{VERSION} ./bin/gpu_{VERSION} {ARGS}

## 5 - Plots

In [ ]:
!pip install -q scipy matplotlib
!python3 scripts/plot_gpu_results.py

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob("results/gpu/graphs/*.png")):
    print(path)
    display(Image(filename=path, width=700))

## 6 - (Optional) Save the results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/bst-parallel-search
!cp -r results/gpu /content/drive/MyDrive/bst-parallel-search/